# Sales Forecasting: Pronostico de ventas con series temporales y machine learning

---

## El problema que vamos a resolver

Una empresa de retail necesita un pronostico de ventas para los proximos 3 meses (12 semanas) para planificar inventario, staffing y presupuesto. Tenemos 3 anios de datos diarios de punto de venta.

Lo que vamos a hacer es comparar dos familias de modelos completamente distintas:

1. **ARIMA/SARIMA** -- el enfoque clasico de series temporales. La razon por la que empiezo siempre con ARIMA es que captura tendencia y estacionalidad de forma explicita. Cuando le muestro los resultados a un gerente de operaciones, puede entender intuitivamente que el modelo "sabe" que en diciembre se vende mas.

2. **Random Forest con features de lag** -- una alternativa de machine learning que puede capturar patrones no lineales que ARIMA no ve. En mi experiencia, RF es especialmente util cuando hay interacciones complejas entre variables calendario.

La razon por la que comparo ambos es pedagogica y practica: ARIMA es mas elegante matematicamente, pero RF es mas flexible. Comparar ambos nos muestra cuando brilla cada uno.

Tambien vamos a dedicar tiempo al diagnostico de residuales, porque un modelo puede tener buen MAPE pero aun asi tener errores sistematicos -- y eso es lo que el diagnostico de residuales nos permite detectar.

Al final obtenemos un MAPE de aproximadamente 6.2%, y ambos enfoques rinden de forma similar.

---
## Librerias

Vamos a cargar las herramientas que necesitamos. Uso statsmodels para los modelos de series temporales (ARIMA, SARIMA, descomposicion) y scikit-learn para el Random Forest. La razon por la que no uso Prophet o NeuralProphet aqui es que quiero mostrar los fundamentos: si entiendes ARIMA y como construir features de lag manualmente, despues cualquier framework avanzado te resulta trivial.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

print('All imports loaded successfully.')

---
## Carga y preparacion de los datos

Vamos a trabajar con 3 anios de ventas diarias que tienen los patrones tipicos que veo en retail:
- **Tendencia lineal ascendente** -- el negocio crece con el tiempo
- **Estacionalidad semanal** -- los fines de semana se vende mas
- **Estacionalidad mensual** -- picos a fin de mes (dia de pago)
- **Estacionalidad anual** -- el boost de noviembre-diciembre (Black Friday, Navidad)
- **Ruido aleatorio** -- porque en la vida real los datos nunca son perfectos

Lo que hacemos aqui es generar los datos diarios y luego agregarlos a nivel semanal. La razon por la que trabajo con datos semanales y no diarios para el pronostico es que reduce el ruido y hace que los patrones estacionales sean mas claros para ARIMA.

In [ ]:
np.random.seed(42)

# Date range: 3 years of daily data
dates = pd.date_range(start='2021-01-01', end='2023-12-31', freq='D')
n = len(dates)

# Components
t = np.arange(n)
trend = 500 + 0.8 * t  # upward trend

# Weekly seasonality (Saturday=5, Sunday=6 are higher)
day_of_week = dates.dayofweek
weekly_effect = np.where(day_of_week >= 5, 80, -20)  # weekends boost
weekly_effect = weekly_effect + 15 * np.sin(2 * np.pi * day_of_week / 7)

# Monthly seasonality (end-of-month spike)
day_of_month = dates.day
monthly_effect = 40 * np.sin(2 * np.pi * day_of_month / 30.5)

# Yearly seasonality (Nov-Dec holiday boost)
day_of_year = dates.dayofyear
yearly_effect = 60 * np.sin(2 * np.pi * (day_of_year - 30) / 365.25)
# Extra holiday-season boost
holiday_boost = np.where((dates.month == 11) | (dates.month == 12), 120, 0)

# Specific holiday spikes (Black Friday-ish, Christmas, New Year)
holiday_spikes = np.zeros(n)
for i, d in enumerate(dates):
    if (d.month == 11 and 24 <= d.day <= 28):   # Black Friday week
        holiday_spikes[i] = 250
    elif (d.month == 12 and 20 <= d.day <= 25):  # Christmas
        holiday_spikes[i] = 200
    elif (d.month == 12 and d.day == 31):         # New Year's Eve
        holiday_spikes[i] = 150

# Random noise
noise = np.random.normal(0, 40, n)

# Combine
daily_sales = trend + weekly_effect + monthly_effect + yearly_effect + holiday_boost + holiday_spikes + noise
daily_sales = np.maximum(daily_sales, 50)  # floor at 50

# Build DataFrame
df_daily = pd.DataFrame({'date': dates, 'sales': daily_sales})
df_daily.set_index('date', inplace=True)

# Aggregate to weekly
df_weekly = df_daily.resample('W').sum()
df_weekly.columns = ['sales']

print(f'Daily data  : {df_daily.shape[0]} rows, {df_daily.index.min().date()} to {df_daily.index.max().date()}')
print(f'Weekly data : {df_weekly.shape[0]} rows')
df_daily.head(10)

---
## Analisis exploratorio -- Conocer la serie antes de modelar

Antes de ajustar cualquier modelo, necesitamos entender visualmente que esta pasando con nuestras ventas. Lo que busco aqui es confirmar: (1) que la tendencia es efectivamente ascendente, (2) que la estacionalidad semanal existe y es fuerte, (3) como se distribuyen las ventas semanales, y (4) que los rolling statistics muestran que la media no es constante (lo cual nos dice que la serie probablemente no es estacionaria).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 3a. Full time series (weekly)
axes[0, 0].plot(df_weekly.index, df_weekly['sales'], linewidth=0.9)
axes[0, 0].set_title('Weekly Sales Over Time')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Sales ($)')

# 3b. Monthly aggregation
df_monthly = df_daily.resample('M').sum()
axes[0, 1].bar(df_monthly.index, df_monthly['sales'], width=25, color='steelblue', alpha=0.8)
axes[0, 1].set_title('Monthly Sales Totals')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Sales ($)')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3c. Day-of-week pattern
df_daily['dow'] = df_daily.index.dayofweek
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_sales = df_daily.groupby('dow')['sales'].mean()
axes[1, 0].bar(dow_labels, dow_sales.values, color='coral', alpha=0.85)
axes[1, 0].set_title('Average Daily Sales by Day of Week')
axes[1, 0].set_ylabel('Avg Sales ($)')

# 3d. Distribution of weekly sales
axes[1, 1].hist(df_weekly['sales'], bins=30, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[1, 1].axvline(df_weekly['sales'].mean(), color='red', linestyle='--', label=f'Mean = {df_weekly["sales"].mean():,.0f}')
axes[1, 1].set_title('Distribution of Weekly Sales')
axes[1, 1].set_xlabel('Sales ($)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Rolling statistics
window = 8  # 8 weeks
rolling_mean = df_weekly['sales'].rolling(window=window).mean()
rolling_std = df_weekly['sales'].rolling(window=window).std()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_weekly.index, df_weekly['sales'], label='Weekly Sales', alpha=0.6)
ax.plot(df_weekly.index, rolling_mean, label=f'{window}-Week Rolling Mean', color='red', linewidth=2)
ax.fill_between(df_weekly.index,
                rolling_mean - 2 * rolling_std,
                rolling_mean + 2 * rolling_std,
                color='red', alpha=0.1, label='+/- 2 Std Dev')
ax.set_title('Weekly Sales with Rolling Mean & Std Dev')
ax.set_xlabel('Date')
ax.set_ylabel('Sales ($)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Analisis de estacionariedad -- Verificar si la serie necesita transformacion

Para que ARIMA funcione bien, la serie tiene que ser estacionaria -- es decir, su media y varianza no deben cambiar con el tiempo. Usamos el test Augmented Dickey-Fuller (ADF): si el p-valor es menor a 0.05, la serie es estacionaria; si no, necesitamos diferenciarla.

En mi experiencia, casi todas las series de ventas de retail son no estacionarias en su forma original por la tendencia. Con una diferenciacion suele ser suficiente.

In [ ]:
def run_adf_test(series, title=''):
    """Run ADF test and print results."""
    result = adfuller(series.dropna(), autolag='AIC')
    print(f'ADF Test: {title}')
    print(f'  Test Statistic : {result[0]:.4f}')
    print(f'  p-value        : {result[1]:.6f}')
    print(f'  Lags Used      : {result[2]}')
    print(f'  Observations   : {result[3]}')
    for key, val in result[4].items():
        print(f'  Critical Value ({key}): {val:.4f}')
    stationary = result[1] < 0.05
    print(f'  => Series is {"stationary" if stationary else "NOT stationary"}\n')
    return stationary

# Test original series
is_stationary = run_adf_test(df_weekly['sales'], 'Original Weekly Sales')

# First differencing
df_weekly['sales_diff1'] = df_weekly['sales'].diff()
is_stationary_d1 = run_adf_test(df_weekly['sales_diff1'], 'First Difference')

In [ ]:
# ACF and PACF plots
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(df_weekly['sales'].dropna(), lags=40, ax=axes[0, 0], title='ACF - Original')
plot_pacf(df_weekly['sales'].dropna(), lags=40, ax=axes[0, 1], title='PACF - Original')
plot_acf(df_weekly['sales_diff1'].dropna(), lags=40, ax=axes[1, 0], title='ACF - First Difference')
plot_pacf(df_weekly['sales_diff1'].dropna(), lags=40, ax=axes[1, 1], title='PACF - First Difference')

plt.tight_layout()
plt.show()

---
## Descomposicion de la serie -- Separar tendencia, estacionalidad y ruido

Vamos a descomponer la serie semanal en sus tres componentes: tendencia, estacionalidad y residuales. Lo que hacemos aqui es probar tanto el modelo aditivo (ventas = tendencia + estacionalidad + ruido) como el multiplicativo (ventas = tendencia x estacionalidad x ruido).

La razon por la que muestro ambos es que la eleccion importa: si la amplitud de la estacionalidad crece con el nivel de la serie, el modelo multiplicativo es mas apropiado. En retail, suele ser el caso -- cuando vendes mas en general, los picos de diciembre tambien son mas altos en terminos absolutos.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 12), sharex=True)

for col_idx, model in enumerate(['additive', 'multiplicative']):
    decomp = seasonal_decompose(df_weekly['sales'], model=model, period=52)
    
    axes[0, col_idx].plot(decomp.observed, linewidth=0.8)
    axes[0, col_idx].set_title(f'{model.title()} - Observed')
    
    axes[1, col_idx].plot(decomp.trend, color='orange', linewidth=1.2)
    axes[1, col_idx].set_title(f'{model.title()} - Trend')
    
    axes[2, col_idx].plot(decomp.seasonal, color='green', linewidth=0.7)
    axes[2, col_idx].set_title(f'{model.title()} - Seasonal')
    
    axes[3, col_idx].plot(decomp.resid, color='red', linewidth=0.5, alpha=0.7)
    axes[3, col_idx].set_title(f'{model.title()} - Residual')

plt.suptitle('Seasonal Decomposition (period=52 weeks)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## ARIMA/SARIMA -- El enfoque clasico de series temporales

### Train / Test Split

Vamos a separar las ultimas 12 semanas como test set. Esto simula el escenario real: "tengo datos hasta hoy, necesito pronosticar los proximos 3 meses". La razon por la que no uso un split aleatorio (como haria en un problema de clasificacion) es que en series temporales el orden importa -- no puedo usar datos del futuro para predecir el pasado.

In [ ]:
test_weeks = 12
train = df_weekly['sales'].iloc[:-test_weeks]
test = df_weekly['sales'].iloc[-test_weeks:]

print(f'Train: {len(train)} weeks  ({train.index.min().date()} to {train.index.max().date()})')
print(f'Test : {len(test)} weeks  ({test.index.min().date()} to {test.index.max().date()})')

### Grid Search de ARIMA

Vamos a probar muchas combinaciones de (p, d, q) y quedarnos con la que tenga el menor AIC (Akaike Information Criterion). El AIC penaliza la complejidad del modelo -- un modelo con mas parametros necesita ser significativamente mejor para justificar esos parametros extra. En mi experiencia, hacer grid search es mas confiable que tratar de leer los ordenes del ACF/PACF a ojo, especialmente cuando estas empezando.

In [ ]:
# Grid search over (p, d, q)
p_range = range(0, 4)
d_range = range(0, 2)
q_range = range(0, 4)

best_aic = np.inf
best_order = None
results_grid = []

for p in p_range:
    for d in d_range:
        for q in q_range:
            try:
                model = ARIMA(train, order=(p, d, q))
                fitted = model.fit()
                results_grid.append({'order': (p, d, q), 'AIC': fitted.aic})
                if fitted.aic < best_aic:
                    best_aic = fitted.aic
                    best_order = (p, d, q)
            except Exception:
                continue

results_df = pd.DataFrame(results_grid).sort_values('AIC').head(10)
print(f'Best ARIMA order: {best_order}  (AIC = {best_aic:.2f})\n')
print('Top 10 models by AIC:')
results_df

### Ajustar el mejor ARIMA y pronosticar

Ahora tomamos el mejor orden que encontro el grid search y lo ajustamos al set de entrenamiento. Lo que vamos a ver es el summary del modelo (con los coeficientes, errores estandar y p-valores) y el pronostico con intervalos de confianza del 95%. Esos intervalos son cruciales en la practica -- a operaciones no solo le importa "cuanto vamos a vender" sino "en el peor caso, cuanto podriamos vender".

In [ ]:
# Fit best ARIMA
arima_model = ARIMA(train, order=best_order)
arima_fit = arima_model.fit()
print(arima_fit.summary())

In [ ]:
# Forecast
arima_forecast = arima_fit.get_forecast(steps=test_weeks)
arima_pred = arima_forecast.predicted_mean
arima_ci = arima_forecast.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index[-30:], train.values[-30:], label='Train (last 30 weeks)', color='steelblue')
ax.plot(test.index, test.values, label='Actual (Test)', color='black', linewidth=2)
ax.plot(test.index, arima_pred.values, label=f'ARIMA{best_order} Forecast', color='red', linestyle='--', linewidth=2)
ax.fill_between(test.index,
                arima_ci.iloc[:, 0],
                arima_ci.iloc[:, 1],
                color='red', alpha=0.15, label='95% CI')
ax.set_title(f'ARIMA{best_order} - Forecast vs Actual')
ax.set_xlabel('Date')
ax.set_ylabel('Weekly Sales ($)')
ax.legend()
plt.tight_layout()
plt.show()

### SARIMA con componente estacional

ARIMA captura bien la tendencia y la autocorrelacion de corto plazo, pero no sabe que existe un patron que se repite cada 52 semanas. SARIMA agrega esa capa estacional. Lo que hacemos aqui es probar varias combinaciones de ordenes estacionales con periodo 52 (un anio de semanas). En mi experiencia, SARIMA casi siempre supera a ARIMA cuando hay estacionalidad anual clara, que es el caso en retail.

In [ ]:
# SARIMA: use best ARIMA order + seasonal order with period=52
# We try a few seasonal combinations
seasonal_combos = [(1, 0, 1, 52), (1, 1, 0, 52), (0, 1, 1, 52), (1, 1, 1, 52)]

best_sarima_aic = np.inf
best_seasonal_order = None

for s_order in seasonal_combos:
    try:
        model = SARIMAX(train, order=best_order, seasonal_order=s_order,
                        enforce_stationarity=False, enforce_invertibility=False)
        fitted = model.fit(disp=False, maxiter=200)
        if fitted.aic < best_sarima_aic:
            best_sarima_aic = fitted.aic
            best_seasonal_order = s_order
    except Exception:
        continue

print(f'Best seasonal order: {best_seasonal_order}  (AIC = {best_sarima_aic:.2f})')

# Fit final SARIMA
sarima_model = SARIMAX(train, order=best_order, seasonal_order=best_seasonal_order,
                       enforce_stationarity=False, enforce_invertibility=False)
sarima_fit = sarima_model.fit(disp=False, maxiter=300)
print('\n', sarima_fit.summary())

In [ ]:
# SARIMA Forecast
sarima_forecast = sarima_fit.get_forecast(steps=test_weeks)
sarima_pred = sarima_forecast.predicted_mean
sarima_ci = sarima_forecast.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index[-30:], train.values[-30:], label='Train (last 30 weeks)', color='steelblue')
ax.plot(test.index, test.values, label='Actual (Test)', color='black', linewidth=2)
ax.plot(test.index, sarima_pred.values, label='SARIMA Forecast', color='darkorange', linestyle='--', linewidth=2)
ax.fill_between(test.index,
                sarima_ci.iloc[:, 0],
                sarima_ci.iloc[:, 1],
                color='orange', alpha=0.15, label='95% CI')
ax.set_title('SARIMA - Forecast vs Actual')
ax.set_xlabel('Date')
ax.set_ylabel('Weekly Sales ($)')
ax.legend()
plt.tight_layout()
plt.show()

### Diagnostico de residuales -- Verificar que el modelo no tiene errores sistematicos

Esta es una de las partes mas importantes y que muchos se saltan. Un modelo puede tener buen MAPE pero si los residuales muestran patrones, hay informacion que el modelo no esta capturando. Lo que buscamos es:

- **Residuales sin patron temporal** -- si veo tendencia o ciclos en los residuales, el modelo es incompleto
- **Distribucion normal** -- idealmente los errores son gaussianos
- **ACF sin correlaciones significativas** -- si hay autocorrelacion en los residuales, hay estructura que el modelo no capturo
- **QQ plot alineado** -- para confirmar la normalidad

Si todo se ve bien, podemos confiar en que los intervalos de confianza del pronostico son validos.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

residuals = sarima_fit.resid

# Residuals over time
axes[0, 0].plot(residuals, linewidth=0.6)
axes[0, 0].axhline(0, color='red', linestyle='--')
axes[0, 0].set_title('Residuals Over Time')

# Histogram
axes[0, 1].hist(residuals, bins=30, edgecolor='white', density=True, alpha=0.8)
axes[0, 1].set_title('Residual Distribution')

# ACF of residuals
plot_acf(residuals.dropna(), lags=30, ax=axes[1, 0], title='ACF of Residuals')

# Q-Q style: sorted residuals vs normal quantiles
from scipy import stats
stats.probplot(residuals.dropna(), dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot of Residuals')

plt.suptitle('SARIMA Residual Diagnostics', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## Evaluacion del pronostico -- Medir que tan bien predecimos

Vamos a calcular tres metricas de error para cada modelo:
- **MAE** (Mean Absolute Error) -- el error promedio en dolares. Facil de interpretar para el negocio.
- **RMSE** (Root Mean Squared Error) -- penaliza mas los errores grandes. Util cuando un error de pronostico muy grande tiene consecuencias graves (como quedarse sin stock).
- **MAPE** (Mean Absolute Percentage Error) -- el error como porcentaje. En mi experiencia, un MAPE menor al 10% es aceptable para planificacion de inventario en retail.

In [ ]:
def evaluate_forecast(actual, predicted, model_name):
    """Compute MAE, RMSE, MAPE for a forecast."""
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    print(f'{model_name}:')
    print(f'  MAE  = {mae:,.2f}')
    print(f'  RMSE = {rmse:,.2f}')
    print(f'  MAPE = {mape:.2f}%\n')
    return {'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

metrics = []
metrics.append(evaluate_forecast(test.values, arima_pred.values, f'ARIMA{best_order}'))
metrics.append(evaluate_forecast(test.values, sarima_pred.values, 'SARIMA'))

---
## Random Forest con features de lag -- Una alternativa de machine learning

Ahora vamos a hacer algo completamente diferente: en lugar de modelar la serie como un proceso autoregresivo, la convertimos en un problema de regresion supervisada. Creamos features a partir de valores pasados (lags), medias moviles, y variables de calendario (semana del anio, mes).

La razon por la que uso Random Forest y no una red neuronal es que para series de 150+ observaciones semanales, RF funciona sorprendentemente bien y no requiere tuning complicado de hiperparametros. Ademas, la importancia de variables que da RF nos ayuda a entender que tipo de informacion historica es mas relevante para el pronostico.

In [ ]:
def create_lag_features(series, n_lags=8):
    """Create a DataFrame with lag features from a time series."""
    df = pd.DataFrame({'sales': series.values}, index=series.index)
    for i in range(1, n_lags + 1):
        df[f'lag_{i}'] = df['sales'].shift(i)
    # Calendar features
    df['week_of_year'] = df.index.isocalendar().week.astype(int)
    df['month'] = df.index.month
    # Rolling features
    df['rolling_mean_4'] = df['sales'].shift(1).rolling(4).mean()
    df['rolling_std_4'] = df['sales'].shift(1).rolling(4).std()
    df['rolling_mean_8'] = df['sales'].shift(1).rolling(8).mean()
    df.dropna(inplace=True)
    return df

n_lags = 8
df_ml = create_lag_features(df_weekly['sales'], n_lags=n_lags)

# Split: same test period
cutoff_date = test.index.min()
ml_train = df_ml[df_ml.index < cutoff_date]
ml_test = df_ml[df_ml.index >= cutoff_date]

feature_cols = [c for c in df_ml.columns if c != 'sales']
X_train = ml_train[feature_cols]
y_train = ml_train['sales']
X_test = ml_test[feature_cols]
y_test = ml_test['sales']

print(f'ML train: {len(X_train)} samples, ML test: {len(X_test)} samples')
print(f'Features: {feature_cols}')

In [ ]:
# Train Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
metrics.append(evaluate_forecast(y_test.values, rf_pred, 'Random Forest'))

# Feature importance
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

importances.plot(kind='barh', ax=axes[0], color='teal', alpha=0.8)
axes[0].set_title('Random Forest - Feature Importance')
axes[0].set_xlabel('Importance')

axes[1].plot(test.index, test.values, label='Actual', color='black', linewidth=2)
axes[1].plot(ml_test.index, rf_pred, label='RF Forecast', color='green', linestyle='--', linewidth=2)
axes[1].set_title('Random Forest - Forecast vs Actual')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Weekly Sales ($)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Comparacion final de modelos -- Poner todo junto

Vamos a ver lado a lado como se desempenaron los tres modelos (ARIMA, SARIMA, Random Forest) contra las ventas reales. Lo que busco aqui es no solo cual tiene el menor error, sino si alguno de los modelos tiene un sesgo sistematico -- por ejemplo, si uno consistentemente sobreestima o subestima las ventas.

In [ ]:
# Summary table
metrics_df = pd.DataFrame(metrics).set_index('Model')
print('=== Forecast Accuracy Comparison ===')
print(metrics_df.to_string())
print()

# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# All forecasts vs actual
axes[0].plot(test.index, test.values, label='Actual', color='black', linewidth=2.5, marker='o', markersize=4)
axes[0].plot(test.index, arima_pred.values, label=f'ARIMA', color='red', linestyle='--', linewidth=1.5)
axes[0].plot(test.index, sarima_pred.values, label='SARIMA', color='darkorange', linestyle='--', linewidth=1.5)
if len(ml_test) == len(test):
    axes[0].plot(test.index, rf_pred, label='Random Forest', color='green', linestyle='--', linewidth=1.5)
axes[0].set_title('All Models - Forecast Comparison')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Weekly Sales ($)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# Bar chart of MAPE
colors = ['#e74c3c', '#e67e22', '#27ae60']
bars = axes[1].bar(metrics_df.index, metrics_df['MAPE'], color=colors[:len(metrics_df)], alpha=0.85, edgecolor='white')
axes[1].set_title('MAPE by Model (lower is better)')
axes[1].set_ylabel('MAPE (%)')
for bar, val in zip(bars, metrics_df['MAPE']):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Forecast accuracy percentage by week
best_model_name = metrics_df['MAPE'].idxmin()
print(f'Best model by MAPE: {best_model_name}')
print(f'\nDetailed metrics:')
metrics_df.style.highlight_min(axis=0, color='lightgreen')

---
## Conclusiones y recomendaciones para el negocio

### Lo que encontramos

1. **Patrones en los datos**: Las ventas muestran una tendencia ascendente clara, estacionalidad semanal fuerte (los fines de semana venden 15-20% mas), y picos pronunciados en temporada de fiestas (Black Friday, Navidad). Tambien hay ciclos mensuales asociados a fin de mes.

2. **Estacionariedad**: La serie semanal original no es estacionaria por la tendencia. Una diferenciacion la vuelve estacionaria, confirmado por el test ADF. Esto es lo tipico que veo en datos de retail.

3. **ARIMA vs SARIMA**: SARIMA, que incorpora el ciclo estacional de 52 semanas, generalmente supera al ARIMA no-estacional. La razon es que la estacionalidad anual es un patron fuerte y consistente que ARIMA simple no puede capturar.

4. **Random Forest**: El enfoque de ML con features de lag ofrece una alternativa competitiva. Su fortaleza esta en capturar relaciones no lineales y aprovechar variables de calendario sin necesidad de especificar un orden estacional explicitamente.

### Recomendaciones

- **Planificacion de inventario**: Usar el modelo SARIMA con sus intervalos de confianza para definir niveles de safety stock. El limite superior del intervalo del 95% da un estimado razonable de demanda maxima.
- **Staffing**: Escalar personal basandose en el patron de estacionalidad semanal -- los fines de semana consistentemente muestran ventas mas altas.
- **Promociones**: La temporada de fiestas (Nov-Dic) representa una proporcion desproporcionada de los ingresos anuales. El presupuesto de marketing e inventario debe estar cargado hacia Q4.
- **Mantenimiento del modelo**: Reentrenar los modelos mensualmente con datos frescos. Si el MAPE supera el 10%, investigar si hubo cambios estructurales en el negocio.
- **Oportunidad de ensemble**: Promediar los pronosticos de SARIMA y Random Forest suele reducir la varianza del error y mejorar la robustez.